# 第2章　確率分布と母関数

統計学実践ワークブック 第2章に対応。  
**方針**：式の形より「何を意味しているのか」のイメージを先に固める。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

plt.rcParams['font.family'] = 'IPAexGothic'
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
import matplotlib
try:
    matplotlib.font_manager.findfont('IPAexGothic', fallback_to_default=False)
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'

np.random.seed(42)

---
## 1. 累積分布関数 (CDF)

**直感：「あなたは下から何番目ですか？」**

模擬試験を受けたとき「あなたの点数は下位 72% に位置します」という表現がある。  
この 72%（= 0.72）が CDF の値。

- CDF $F(x)$ ＝「$x$ 以下の値が出る確率」
- 必ず 0 から始まり 1 で終わる（全員を数えれば 100%）
- 左から右に単調に増加する（順位は逆転しない）

In [ ]:
# 1000人分の模擬試験スコア（平均60点、標準偏差15点）
scores = np.random.normal(loc=60, scale=15, size=1000)
scores = np.clip(scores, 0, 100)

my_score = 72  # 自分のスコア

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左：ヒストグラム + 自分の位置
ax1 = axes[0]
ax1.hist(scores, bins=30, color='#457b9d', alpha=0.7, edgecolor='white')
ax1.axvline(my_score, color='#e63946', lw=2.5, label=f'自分のスコア: {my_score}点')
# 自分以下を塗る
below = scores[scores <= my_score]
ax1.hist(below, bins=30, color='#e63946', alpha=0.4, edgecolor='white',
         label=f'自分以下: {len(below)/len(scores)*100:.1f}%')
ax1.set_xlabel('点数', fontsize=11)
ax1.set_ylabel('人数', fontsize=11)
ax1.set_title('1000人の試験スコア分布', fontsize=11)
ax1.legend(fontsize=9)

# 右：CDF（経験分布 + 理論曲線）
ax2 = axes[1]
sorted_scores = np.sort(scores)
cdf_empirical = np.arange(1, len(sorted_scores)+1) / len(sorted_scores)
ax2.plot(sorted_scores, cdf_empirical, color='#457b9d', lw=1.5,
         alpha=0.7, label='経験 CDF（実データ）')

x_range = np.linspace(0, 100, 300)
ax2.plot(x_range, stats.norm.cdf(x_range, 60, 15),
         color='#e63946', lw=2.5, label='理論 CDF')

# 自分のスコアに対応する CDF 値
cdf_mine = stats.norm.cdf(my_score, 60, 15)
ax2.scatter([my_score], [cdf_mine], color='#e63946', s=100, zorder=5)
ax2.plot([my_score, my_score], [0, cdf_mine], color='gray', lw=1.5, linestyle='--')
ax2.plot([0, my_score], [cdf_mine, cdf_mine], color='gray', lw=1.5, linestyle='--')
ax2.text(my_score + 1, cdf_mine - 0.06,
         f'F({my_score}) = {cdf_mine:.2f}\n→ 下位 {cdf_mine*100:.0f}%',
         fontsize=9, color='#e63946')

ax2.set_xlabel('点数 x', fontsize=11)
ax2.set_ylabel('F(x) = P(スコア ≤ x)', fontsize=11)
ax2.set_title('CDF：「x 点以下は何%いるか」', fontsize=11)
ax2.legend(fontsize=9)

plt.suptitle('累積分布関数 (CDF) = パーセンタイルそのもの', fontsize=13)
plt.tight_layout()
plt.show()

print(f'72点の人は下位 {stats.norm.cdf(72, 60, 15)*100:.1f}%')
print(f'上位10%に入るには {stats.norm.ppf(0.9, 60, 15):.1f}点以上必要')

---
## 2. 生存関数

**直感：「買ってから x 時間後、まだ動いている確率」**

電球を100個買ったとして、時間が経つにつれて1つずつ切れていく。  
「x 時間後にまだ光っている割合」が生存関数 $S(x)$。

- $S(x) = 1 - F(x) = P(X > x)$  
- CDF と鏡写し：CDF が上がるにつれて生存関数は下がる
- 必ず 1 から始まり 0 に向かって下がっていく

In [ ]:
# 電球100個の寿命をシミュレーション（指数分布、平均1000時間）
n_bulbs = 100
mean_life = 1000  # 時間
lifetimes = np.random.exponential(mean_life, n_bulbs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左：時間経過で電球が消えていく様子
ax1 = axes[0]
t_points = np.linspace(0, 4000, 200)
surviving = np.array([np.sum(lifetimes > t) for t in t_points])

ax1.step(t_points, surviving, color='#f4a261', lw=2.5, where='post')
ax1.fill_between(t_points, surviving, step='post', alpha=0.2, color='#f4a261')

# チェックポイント
for t_check, col in [(500, '#e63946'), (1000, '#457b9d'), (2000, '#2a9d8f')]:
    n_alive = np.sum(lifetimes > t_check)
    ax1.scatter([t_check], [n_alive], s=80, color=col, zorder=5)
    ax1.annotate(f'{t_check}h後\n{n_alive}個生存',
                 xy=(t_check, n_alive), xytext=(t_check+200, n_alive+5),
                 fontsize=8, color=col)

ax1.set_xlabel('時間 (h)', fontsize=11)
ax1.set_ylabel('生存している電球の数', fontsize=11)
ax1.set_title(f'電球 {n_bulbs} 個の生存推移（平均寿命 {mean_life}h）', fontsize=11)

# 右：生存関数 vs CDF
ax2 = axes[1]
t_range = np.linspace(0, 5000, 500)
S = np.exp(-t_range / mean_life)        # 指数分布の生存関数
F = 1 - S                               # CDF

ax2.plot(t_range, S, color='#2a9d8f', lw=2.5, label='生存関数 S(t) = P(X > t)')
ax2.plot(t_range, F, color='#e63946', lw=2.5, linestyle='--',
         label='CDF F(t) = P(X ≤ t)')
ax2.axhline(0.5, color='gray', lw=1, linestyle=':')
ax2.axvline(mean_life * np.log(2), color='gray', lw=1, linestyle=':')
ax2.text(mean_life*np.log(2)+50, 0.52,
         f'半減期\n≈ {mean_life*np.log(2):.0f}h', fontsize=8, color='gray')

ax2.set_xlabel('時間 t (h)', fontsize=11)
ax2.set_ylabel('確率', fontsize=11)
ax2.set_title('生存関数と CDF は鏡写し\nS(t) + F(t) = 1', fontsize=11)
ax2.legend(fontsize=9)

plt.suptitle('生存関数 S(t)：「まだ生きている確率」', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. ハザード関数

**直感：「今この瞬間、壊れる危険度はどのくらいか」**

生存関数は「まだ生きている割合」だが、ハザード関数は視点が違う。  
**「今まで生き残ってきた」という条件の下で、今この瞬間に壊れる確率の密度**。

工業製品の故障パターンを表す**バスタブ曲線**が典型例：

| 期間 | ハザード | 意味 |
|---|---|---|
| 初期（購入直後） | 高い→下がる | 初期不良品がすぐ壊れる |
| 安定期（中間） | 一定（低い） | ランダムな偶発故障のみ |
| 老化期（末期） | 上がる | 磨耗・老化で壊れやすくなる |

In [ ]:
t = np.linspace(0.01, 5, 500)

# ワイブル分布でバスタブ曲線の3つのフェーズを表現
# ハザード h(t) = (c/λ)(t/λ)^(c-1)
phases = [
    (0.5, '#e63946', '初期故障型 (c<1)\nハザードが減少'),
    (1.0, '#457b9d', '安定期・偶発故障型 (c=1)\nハザードが一定（指数分布）'),
    (3.5, '#2a9d8f', '老化・磨耗型 (c>1)\nハザードが増加'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 左：3タイプのハザード関数
ax1 = axes[0]
for c, col, lbl in phases:
    # ワイブル c=shape, scale=1 の場合 h(t) = c * t^(c-1)
    h = stats.weibull_min(c).pdf(t) / (1 - stats.weibull_min(c).cdf(t) + 1e-10)
    ax1.plot(t, h, color=col, lw=2.5, label=lbl)

ax1.set_ylim(0, 4)
ax1.set_xlabel('時間 t', fontsize=10)
ax1.set_ylabel('ハザード h(t)', fontsize=10)
ax1.set_title('3種類の故障パターン', fontsize=11)
ax1.legend(fontsize=7.5)

# 中：バスタブ曲線（3フェーズの合成）
ax2 = axes[1]
t2 = np.linspace(0.01, 10, 500)
# 初期故障 + 安定期 + 老化の合成
h_early   = 0.5 * stats.weibull_min(0.5).pdf(t2) / (stats.weibull_min(0.5).sf(t2) + 1e-10)
h_random  = np.full_like(t2, 0.1)
h_wearout = 0.3 * stats.weibull_min(4.0, scale=8).pdf(t2) / (stats.weibull_min(4.0, scale=8).sf(t2) + 1e-10)
h_bath    = h_early + h_random + h_wearout

ax2.plot(t2, h_bath, color='#6a4c93', lw=3)
ax2.fill_between(t2[:120], h_bath[:120], alpha=0.2, color='#e63946', label='初期故障期')
ax2.fill_between(t2[120:350], h_bath[120:350], alpha=0.2, color='#457b9d', label='安定期（偶発故障）')
ax2.fill_between(t2[350:], h_bath[350:], alpha=0.2, color='#2a9d8f', label='老化期')
ax2.set_ylim(0, 2)
ax2.set_xlabel('時間 t', fontsize=10)
ax2.set_ylabel('ハザード h(t)', fontsize=10)
ax2.set_title('バスタブ曲線\n（実際の製品故障のイメージ）', fontsize=11)
ax2.legend(fontsize=8)

# 右：生存関数・PDF・ハザードの関係を1つの分布で確認
ax3 = axes[2]
c_demo = 2.5
dist = stats.weibull_min(c_demo)
f_d = dist.pdf(t)
S_d = dist.sf(t)
h_d = f_d / (S_d + 1e-10)

ax3.plot(t, f_d / f_d.max(), color='#457b9d', lw=2, label='PDF f(t)（正規化）')
ax3.plot(t, S_d, color='#2a9d8f', lw=2, label='生存関数 S(t)')
ax3.plot(t, h_d / h_d.max(), color='#e63946', lw=2, linestyle='--',
         label='ハザード h(t)（正規化）')
ax3.set_xlabel('時間 t', fontsize=10)
ax3.set_title('f, S, h の関係\nh(t) = f(t) / S(t)', fontsize=11)
ax3.legend(fontsize=8)

plt.suptitle('ハザード関数：「今まで生き残ってきた前提で今の危険度」', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. 同時確率関数・同時確率密度関数

**直感：「2つの変数を同時に記録したときの分布」**

1変数の分布は「ヒストグラム（棒グラフ）」でイメージできる。  
2変数になると「地形図（等高線マップ）」になる。

- **離散型（同時確率関数）**：表のマス目それぞれに確率が入る。全マスの合計 = 1
- **連続型（同時確率密度関数）**：2次元の山。山全体の体積 = 1

**重要な注意**：2変数が**相関している**とき、山は斜めに傾く。  
→ 「身長が高い人は体重も重い傾向がある」

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- 離散型：サイコロ2個 ---
ax1 = axes[0]
# 各目の組み合わせ確率 = 1/36
table = np.full((6, 6), 1/36)
im = ax1.imshow(table, cmap='Blues', vmin=0, vmax=1/18)
for i in range(6):
    for j in range(6):
        ax1.text(j, i, '1/36', ha='center', va='center', fontsize=7)
ax1.set_xticks(range(6)); ax1.set_xticklabels(range(1, 7))
ax1.set_yticks(range(6)); ax1.set_yticklabels(range(1, 7))
ax1.set_xlabel('サイコロ1 (X)', fontsize=10)
ax1.set_ylabel('サイコロ2 (Y)', fontsize=10)
ax1.set_title('離散型：同時確率関数\n全36マス×(1/36) = 1', fontsize=10)

# --- 連続型：身長・体重（相関あり）---
# 身長: 平均170, SD=7 / 体重: 平均65, SD=10 / 相関0.6
mu = [170, 65]
cov = [[7**2, 0.6*7*10], [0.6*7*10, 10**2]]
rv = stats.multivariate_normal(mean=mu, cov=cov)

h_grid = np.linspace(145, 195, 100)
w_grid = np.linspace(35, 95, 100)
H, W = np.meshgrid(h_grid, w_grid)
Z = rv.pdf(np.dstack((H, W)))

ax2 = axes[1]
cf = ax2.contourf(H, W, Z, levels=12, cmap='RdYlBu_r')
ax2.contour(H, W, Z, levels=12, colors='white', linewidths=0.5, alpha=0.4)
plt.colorbar(cf, ax=ax2, shrink=0.8, label='密度')
ax2.set_xlabel('身長 (cm)', fontsize=10)
ax2.set_ylabel('体重 (kg)', fontsize=10)
ax2.set_title('連続型：同時確率密度関数\n（身長と体重、相関ρ=0.6）', fontsize=10)

# --- 相関なしとの比較 ---
cov_indep = [[7**2, 0], [0, 10**2]]
rv_indep = stats.multivariate_normal(mean=mu, cov=cov_indep)
Z_indep = rv_indep.pdf(np.dstack((H, W)))

ax3 = axes[2]
ax3.contourf(H, W, Z_indep, levels=12, cmap='RdYlBu_r')
ax3.contour(H, W, Z_indep, levels=12, colors='white', linewidths=0.5, alpha=0.4)
ax3.set_xlabel('身長 (cm)', fontsize=10)
ax3.set_ylabel('体重 (kg)', fontsize=10)
ax3.set_title('相関なし（ρ=0）の場合\n楕円が軸に平行になる', fontsize=10)

plt.suptitle('同時分布：「2変数を同時に記録した地形図」\n相関があると楕円が斜めに傾く', fontsize=12)
plt.tight_layout()
plt.show()

---
## 5. 周辺分布・周辺確率関数・周辺確率密度関数

**直感：「もう片方の変数を無視したら何が残るか」**

身長と体重を同時に記録したデータがある。  
「体重のことはどうでもいい、身長だけを見たい」→ これが**周辺分布**。

操作のイメージ：
- 離散型：表の行（または列）を**合計**する
- 連続型：もう片方の変数方向に**積み重ねる**（積分する）

2次元の地形図を「押しつぶして」1次元のヒストグラムにするイメージ。

In [ ]:
# 身長・体重データを実際に生成して可視化
n = 2000
mu = [170, 65]
cov = [[7**2, 0.6*7*10], [0.6*7*10, 10**2]]
data = np.random.multivariate_normal(mu, cov, n)
heights = data[:, 0]
weights = data[:, 1]

fig = plt.figure(figsize=(10, 8))
gs = fig.add_gridspec(2, 2, width_ratios=[3, 1.2], height_ratios=[1.2, 3],
                      hspace=0.05, wspace=0.05)

ax_main  = fig.add_subplot(gs[1, 0])
ax_top   = fig.add_subplot(gs[0, 0], sharex=ax_main)
ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)

# 中央：散布図（同時分布）
ax_main.scatter(heights, weights, alpha=0.08, s=10, color='#457b9d')
# 等高線も重ねる
h_grid = np.linspace(145, 195, 80)
w_grid = np.linspace(35, 95, 80)
H, W = np.meshgrid(h_grid, w_grid)
rv = stats.multivariate_normal(mean=mu, cov=cov)
ax_main.contour(H, W, rv.pdf(np.dstack((H, W))), levels=6,
                colors='#e63946', linewidths=1.5, alpha=0.7)
ax_main.set_xlabel('身長 (cm)', fontsize=11)
ax_main.set_ylabel('体重 (kg)', fontsize=11)
ax_main.text(0.05, 0.95, '同時分布\nf(身長, 体重)',
             transform=ax_main.transAxes, fontsize=9,
             va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

# 上：身長の周辺分布（体重を無視）
ax_top.hist(heights, bins=40, color='#e63946', alpha=0.7, density=True,
            edgecolor='white', label='体重を無視した\n身長の分布')
h_range = np.linspace(145, 195, 200)
ax_top.plot(h_range, stats.norm.pdf(h_range, mu[0], 7),
            'k--', lw=1.5, alpha=0.7, label='理論値')
ax_top.set_ylabel('密度', fontsize=9)
ax_top.set_title('周辺分布（身長）：体重を無視 → 1次元のヒストグラムに', fontsize=10)
plt.setp(ax_top.get_xticklabels(), visible=False)
ax_top.legend(fontsize=7, loc='upper right')

# 右：体重の周辺分布（身長を無視）
ax_right.hist(weights, bins=40, color='#2a9d8f', alpha=0.7, density=True,
              orientation='horizontal', edgecolor='white')
w_range = np.linspace(35, 95, 200)
ax_right.plot(stats.norm.pdf(w_range, mu[1], 10), w_range,
              'k--', lw=1.5, alpha=0.7)
ax_right.set_xlabel('密度', fontsize=9)
plt.setp(ax_right.get_yticklabels(), visible=False)
ax_right.set_title('周辺分布\n（体重）', fontsize=9)

fig.add_subplot(gs[0, 1]).axis('off')

plt.suptitle('周辺分布：2次元を「潰して」1次元に', fontsize=13, y=1.01)
plt.show()

print(f'身長の周辺分布: 平均={heights.mean():.1f}cm, SD={heights.std():.1f}cm')
print(f'体重の周辺分布: 平均={weights.mean():.1f}kg, SD={weights.std():.1f}kg')
print('→ 相関があっても、周辺分布は各変数それぞれの正規分布になる')

---
## 6. 条件付き確率関数・条件付き確率密度関数

**直感：「特定のグループに絞ったときの分布」**

「全員の体重分布」ではなく「**身長170cmの人だけ**の体重分布」を知りたい。  
→ これが条件付き分布。

操作のイメージ：
- データを**フィルタリング**して部分集合を取り出す
- その部分集合の分布を見る（面積が1になるように正規化する）

**周辺分布との違い**：
- 周辺分布：片方を無視する（捨てる）
- 条件付き分布：片方の値を固定する（絞り込む）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：散布図に「身長 x の条件」を線で示す
ax1 = axes[0]
ax1.scatter(heights, weights, alpha=0.1, s=10, color='#457b9d')

height_conditions = [160, 170, 180]
colors_cond = ['#e63946', '#2a9d8f', '#f4a261']
band = 3  # ±3cm のバンドでフィルタ

for h_cond, col in zip(height_conditions, colors_cond):
    ax1.axvspan(h_cond - band, h_cond + band, alpha=0.2, color=col,
                label=f'身長 {h_cond}±{band}cm')
    ax1.axvline(h_cond, color=col, lw=1.5, linestyle='--')

ax1.set_xlabel('身長 (cm)', fontsize=11)
ax1.set_ylabel('体重 (kg)', fontsize=11)
ax1.set_title('条件 = 「この帯に入るデータだけ見る」', fontsize=11)
ax1.legend(fontsize=8)

# 右：各条件での体重の分布
ax2 = axes[1]
w_range = np.linspace(30, 100, 300)

for h_cond, col in zip(height_conditions, colors_cond):
    # 2変量正規の条件付き分布: X|Y=y ~ N(μ_x + ρ*(σ_x/σ_y)*(y-μ_y), σ_x²(1-ρ²))
    rho = 0.6
    sig_h, sig_w = 7, 10
    mu_w_given_h = mu[1] + rho * (sig_w / sig_h) * (h_cond - mu[0])
    sig_w_given_h = sig_w * np.sqrt(1 - rho**2)

    ax2.plot(w_range, stats.norm.pdf(w_range, mu_w_given_h, sig_w_given_h),
             color=col, lw=2.5,
             label=f'身長={h_cond}cm → 体重の平均={mu_w_given_h:.1f}kg')
    ax2.axvline(mu_w_given_h, color=col, lw=1, linestyle=':')

    # データからも確認
    mask = (heights >= h_cond - band) & (heights <= h_cond + band)
    ax2.hist(weights[mask], bins=15, density=True, alpha=0.2, color=col)

ax2.set_xlabel('体重 (kg)', fontsize=11)
ax2.set_ylabel('条件付き密度', fontsize=11)
ax2.set_title('条件付き分布：身長が高いほど\n体重の平均も右にシフト', fontsize=11)
ax2.legend(fontsize=8)

plt.suptitle('条件付き分布：「特定グループに絞り込んだときの分布」', fontsize=12)
plt.tight_layout()
plt.show()

print('条件付き平均 E[体重 | 身長=h] = μ_w + ρ*(σ_w/σ_h)*(h - μ_h)')
for h_c in height_conditions:
    mu_w_c = mu[1] + 0.6 * (10/7) * (h_c - mu[0])
    print(f'  身長={h_c}cm → 体重の条件付き平均 = {mu_w_c:.1f}kg')

---
## 7. モーメント母関数 (MGF)

**直感：「分布の全情報を1つの関数に圧縮した指紋」**

分布の特徴を表す数値（モーメント）には以下がある：

| 名前 | 意味 |
|---|---|
| 1次モーメント = $E[X]$ | 平均（重心） |
| 2次モーメント = $E[X^2]$ | 分散の計算素材 |
| 3次中心モーメント | 歪み（右に長い尾 or 左に長い尾） |
| 4次中心モーメント | 尖り（裾が重いか軽いか） |

MGF はこれらを**一気に生成**するための関数。  
$t=0$ まわりで微分すると、どんな次数のモーメントも取り出せる。

**エンジニア的なたとえ**：クラスの中に「任意のメソッドを呼ぶとそのモーメントを返す」オブジェクトがあるイメージ。

In [ ]:

# scipy.misc.derivative は非推奨のため、手動で数値微分を実装
def num_deriv(f, x, n=1, h=1e-4):
    """中心差分による n 階数値微分"""
    if n == 1:
        return (f(x + h) - f(x - h)) / (2 * h)
    elif n == 2:
        return (f(x + h) - 2*f(x) + f(x - h)) / h**2
    elif n == 3:
        return (f(x + 2*h) - 2*f(x + h) + 2*f(x - h) - f(x - 2*h)) / (2 * h**3)

# ポアソン分布 λ=3 の MGF: M(t) = exp(λ(e^t - 1))
lam = 3.0

def mgf_poisson(t):
    return np.exp(lam * (np.exp(t) - 1))

E1 = num_deriv(mgf_poisson, 0.0, n=1)   # 1階微分 → E[X]
E2 = num_deriv(mgf_poisson, 0.0, n=2)   # 2階微分 → E[X²]
E3 = num_deriv(mgf_poisson, 0.0, n=3)   # 3階微分 → E[X³]

print('=== MGF を微分してモーメントを取り出す（ポアソン λ=3）===')
print(f"M'(0)  = E[X]   = {E1:.3f}  （理論値 λ={lam:.1f}）")
print(f"M''(0) = E[X²]  = {E2:.3f}  （理論値 λ²+λ={lam**2+lam:.1f}）")
print(f"Var(X) = E[X²] - E[X]² = {E2 - E1**2:.3f}  （理論値 λ={lam:.1f}）")

# MGF の形を可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

t_range = np.linspace(-1.5, 1.5, 300)

# 左：MGF のグラフ（λを変えたとき）
ax1 = axes[0]
for lam_v, col in [(1, '#457b9d'), (3, '#e63946'), (6, '#2a9d8f')]:
    ax1.plot(t_range, np.exp(lam_v*(np.exp(t_range)-1)),
             color=col, lw=2, label=f'λ={lam_v}')
ax1.scatter([0], [1], color='black', s=80, zorder=5)
ax1.text(0.05, 1.2, 'M(0)=1\n（t=0では必ず1）', fontsize=9)
ax1.set_xlabel('t', fontsize=11)
ax1.set_ylabel('M(t)', fontsize=11)
ax1.set_ylim(0, 30); ax1.legend(fontsize=10)
ax1.set_title('ポアソン分布の MGF\nλが大きいほど急激に増大', fontsize=11)

# 右：t=0 周りの傾き = E[X] を視覚化
ax2 = axes[1]
t_zoom = np.linspace(-0.5, 0.5, 200)
ax2.plot(t_zoom, mgf_poisson(t_zoom), color='#e63946', lw=2.5, label='M(t)')

# t=0 での接線（傾き = E[X] = λ = 3）
tangent = 1 + lam * t_zoom   # M(0) + M'(0)*t
ax2.plot(t_zoom, tangent, 'k--', lw=1.5, alpha=0.7,
         label=f"接線の傾き = E[X] = λ = {lam:.0f}")
ax2.scatter([0], [1], color='black', s=80, zorder=5)
ax2.set_xlabel('t', fontsize=11)
ax2.set_ylabel('M(t)', fontsize=11)
ax2.legend(fontsize=9)
ax2.set_title('t=0 での接線の傾き = E[X]\n（1階微分がモーメントを生成）', fontsize=11)

plt.suptitle('モーメント母関数 (MGF)：微分するだけで平均・分散・高次モーメントが出てくる', fontsize=11)
plt.tight_layout()
plt.show()


---
## 8. 特性関数

**直感：「MGF の弱点を克服した上位互換」**

MGF の弱点：**裾が重い分布**（外れ値が多い分布）では $E[e^{tX}]$ が無限大に発散して定義できない。  
→ コーシー分布など、実際によく使う分布で使えないことがある。

特性関数は $e^{tX}$ の代わりに $e^{itX}$（複素数）を使う。  
$|e^{itX}| = 1$ なので**絶対値は常に1**。どんな分布でも必ず収束する。

**フーリエ変換との関係**：特性関数は PDF のフーリエ変換そのもの。  
→ 信号処理をやっていたエンジニアには馴染みのある変換の統計版。

| | MGF | 特性関数 |
|---|---|---|
| 定義 | $E[e^{tX}]$（$t$ は実数） | $E[e^{itX}]$（$t$ は実数、$i$ は虚数単位） |
| 存在 | 分布によっては発散 | **常に存在** |
| 主な用途 | モーメント計算 | 分布の一意識別・CLT の証明 |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

t_vals = np.linspace(-6, 6, 400)

# --- 左：MGF が発散する例 vs 収束する例 ---
ax1 = axes[0]
# 正規分布の MGF は存在する
t_safe = np.linspace(-2, 2, 200)
ax1.plot(t_safe, np.exp(0.5 * t_safe**2), color='#2a9d8f', lw=2,
         label='正規分布 N(0,1)\nMGF = exp(t²/2) ✓ 存在する')

# コーシー分布の MGF は存在しない（シミュレーションで確認）
n_cauchy = 5000
cauchy_samples = stats.cauchy.rvs(size=n_cauchy)
t_pos = 0.3
mgf_estimate = np.mean(np.exp(t_pos * cauchy_samples))
ax1.text(0, 3, f'コーシー分布の MGF:\nE[e^{{0.3X}}] ≈ {mgf_estimate:.1f}\n（外れ値のせいで発散）',
         fontsize=9, color='#e63946',
         bbox=dict(boxstyle='round', facecolor='#ffe5e5', alpha=0.8))

ax1.set_xlabel('t', fontsize=11); ax1.set_ylabel('M(t)', fontsize=11)
ax1.set_title('MGF は一部の分布で発散する', fontsize=10)
ax1.legend(fontsize=8); ax1.set_ylim(0, 8)

# --- 中：特性関数の実部（正規分布）---
ax2 = axes[1]
# φ(t) = exp(-t²/2) は実数値（対称分布なので虚部=0）
for mu_v, sig_v, col, lbl in [
    (0, 1,   '#e63946', 'N(0,1)'),
    (0, 0.5, '#457b9d', 'N(0,0.25)\nσ小→早く減衰'),
    (2, 1,   '#2a9d8f', 'N(2,1)\nμ≠0→振動'),
]:
    re_phi = np.exp(-0.5 * sig_v**2 * t_vals**2) * np.cos(mu_v * t_vals)
    ax2.plot(t_vals, re_phi, color=col, lw=2, label=lbl)

ax2.axhline(0, color='black', lw=0.8, linestyle='--')
ax2.set_xlabel('t', fontsize=11); ax2.set_ylabel('Re[φ(t)]', fontsize=11)
ax2.set_title('特性関数の実部\n|φ(t)| ≤ 1 → 常に有界', fontsize=10)
ax2.legend(fontsize=8)

# --- 右：コーシー分布の特性関数は存在する ---
ax3 = axes[2]
# コーシー分布の特性関数: φ(t) = exp(-|t|) （解析式）
phi_cauchy = np.exp(-np.abs(t_vals))
phi_normal = np.exp(-0.5 * t_vals**2)

ax3.plot(t_vals, phi_cauchy, color='#e63946', lw=2.5,
         label='コーシー分布\nφ(t) = exp(-|t|) ✓ 存在する')
ax3.plot(t_vals, phi_normal, color='#457b9d', lw=2.5, linestyle='--',
         label='正規分布 N(0,1)\nφ(t) = exp(-t²/2)')
ax3.set_xlabel('t', fontsize=11); ax3.set_ylabel('φ(t)', fontsize=11)
ax3.set_title('MGF が存在しないコーシー分布も\n特性関数は必ず存在する', fontsize=10)
ax3.legend(fontsize=9)

plt.suptitle('特性関数 φ(t) = E[e^{itX}]：|e^{itX}|=1 だから常に有界、どんな分布でも定義できる', fontsize=11)
plt.tight_layout()
plt.show()

print('フーリエ変換との対応：')
print('  特性関数 φ(t) = ∫ e^{itx} f(x) dx  ← f(x) のフーリエ変換')
print('  逆フーリエ変換で f(x) を復元できる → 特性関数が一致すれば分布も一致する')

---
## まとめ

| 概念 | 直感的な意味 |
|---|---|
| CDF | 「x 以下は何%か」＝パーセンタイル |
| 生存関数 | 「時刻 x を超えてまだ生きている確率」、CDF の裏返し |
| ハザード関数 | 「今まで生き残ってきた前提で、今この瞬間の危険度」 |
| 同時分布 | 2変数を同時に記録した「地形図」 |
| 周辺分布 | 片方を無視して1次元に潰す |
| 条件付き分布 | 片方を固定値に絞り込む |
| MGF | 微分するたびにモーメントが出てくる「指紋関数」 |
| 特性関数 | MGF の上位互換。複素数を使うため常に存在。PDF のフーリエ変換 |

**次章**：第3章「分布の特性値」（平均・分散・歪度・尖度・共分散・相関係数）